<a href="https://colab.research.google.com/github/testoumut/movie-recommender-svd/blob/main/notebooks/02_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Milestone 2: Honest Testing and Baselines to Beat
## Part A: Load the MovieLens ratings
First, we import the required libraries (`os`, `urllib.request`, `zipfile`, `numpy`, and `pandas`), download and unzip the MovieLens 100K dataset into the Colab workspace if it is not already downloaded, and read the tab-separated `u.data` file into a DataFrame containing the user ID, movie ID, star rating, and timestamp.

In [1]:
import os, urllib.request, zipfile
import numpy as np
import pandas as pd

if not os.path.exists("ml-100k"):
    urllib.request.urlretrieve(
        "https://files.grouplens.org/datasets/movielens/ml-100k.zip",
        "ml-100k.zip")
    zipfile.ZipFile("ml-100k.zip").extractall()

ratings = pd.read_csv(
    "ml-100k/u.data", sep="\t",
    names=["user", "item", "rating", "time"])

print(ratings.head())
print("Number of ratings:", len(ratings))

   user  item  rating       time
0   196   242       3  881250949
1   186   302       3  891717742
2    22   377       1  878887116
3   244    51       2  880606923
4   166   346       1  886397596
Number of ratings: 100000


## Part B: Make an honest train/test split
This cell defines `split_ratings`, which visits all 100,000 ratings in a reproducible random order (`seed=0`) and places roughly 10% of them into a hidden test set while guaranteeing that every user and every movie keeps at least one rating in the training set. Keeping the test set strictly separate prevents data leakage.

In [2]:
def split_ratings(ratings, test_frac=0.10, seed=0):
    """Hide ratings without removing a user's or movie's last training rating."""
    rng = np.random.default_rng(seed)
    is_test = np.zeros(len(ratings), dtype=bool)
    user_left = ratings.groupby("user").size().to_dict()
    item_left = ratings.groupby("item").size().to_dict()

    for pos in rng.permutation(len(ratings)):
        row = ratings.iloc[pos]
        user, item = row["user"], row["item"]
        if (rng.random() < test_frac
                and user_left[user] > 1
                and item_left[item] > 1):
            is_test[pos] = True
            user_left[user] -= 1
            item_left[item] -= 1

    return ratings.iloc[~is_test].copy(), ratings.iloc[is_test].copy()

train, test = split_ratings(ratings)
print("train:", len(train), "test:", len(test))
print("test fraction:", round(len(test) / len(ratings), 3))

train: 90032 test: 9968
test fraction: 0.1


## Part C: Build the training matrix
Here we convert the training DataFrame into a $943 \times 1682$ user-by-movie matrix `R` where unobserved entries are stored as `np.nan` (subtracting 1 from the 1-based user and movie IDs to match Python's 0-based indexing). We also extract the true hidden star ratings (`truth`) and their zero-based row (`u`) and column (`i`) coordinates from the test set.

In [3]:
N_USERS, N_ITEMS = 943, 1682

def to_matrix(df):
    """Return a user-by-movie matrix; missing ratings are np.nan."""
    R = np.full((N_USERS, N_ITEMS), np.nan)
    R[df["user"].to_numpy() - 1,
      df["item"].to_numpy() - 1] = df["rating"].to_numpy()
    return R

R = to_matrix(train)
known = ~np.isnan(R)
print("Matrix shape:", R.shape)
print("Known training ratings:", known.sum())

truth = test["rating"].to_numpy()
u = test["user"].to_numpy() - 1
i = test["item"].to_numpy() - 1

Matrix shape: (943, 1682)
Known training ratings: 90032


## Part D: Score predictions with RMSE
We define the `rmse` function to compute the Root Mean Squared Error between predicted ratings and actual hidden test ratings:
$$\text{RMSE} = \sqrt{\frac{1}{N}\sum_{j=1}^{N}(\text{prediction}_j - \text{truth}_j)^2}$$
A smaller RMSE indicates more accurate predictions measured in stars.

In [4]:
def rmse(pred, truth):
    """Root mean squared error, measured in stars."""
    return np.sqrt(np.mean((pred - truth)**2))

## Part E: Build simple baselines
We calculate the overall training average (`mu`), each user's average training rating (`user_mean`), and each movie's average training rating (`item_mean`, using `mu` as a fallback if a movie has no training ratings). We then evaluate four simple baselines on the hidden test ratings: always predicting 3 stars, predicting the overall average, predicting the user average, and predicting the movie average.

In [5]:
mu = np.nanmean(R)
user_mean = np.nanmean(R, axis=1)

n_item = known.sum(axis=0)
item_mean = np.full(N_ITEMS, mu)
has_rating = n_item > 0
item_mean[has_rating] = (
    np.nansum(R, axis=0)[has_rating] / n_item[has_rating])

print("Overall average:", round(mu, 3))

scores = {}
scores["always 3"] = rmse(np.full(len(test), 3.0), truth)
scores["overall average"] = rmse(np.full(len(test), mu), truth)
scores["user average"] = rmse(user_mean[u], truth)
scores["movie average"] = rmse(item_mean[i], truth)

for name, score in scores.items():
    print(f"{name:16s} {score:.4f}")

Overall average: 3.53
always 3         1.2516
overall average  1.1349
user average     1.0426
movie average    1.0284


## Part F: Combine user and movie effects
The `bias_baseline` function captures both user generosity and movie popularity through the additive model $\text{prediction} = \mu + b_{\text{user}} + b_{\text{item}}$. First, `b_user` measures each user's average deviation from `mu`; next, `b_item` averages the remaining residuals for each movie column. After clipping predictions to the valid $[1, 5]$ star range, this combined baseline achieves our best benchmark RMSE of `0.9545`.

In [6]:
def bias_baseline(R):
    """Overall mean plus one user effect and one movie effect."""
    known = ~np.isnan(R)
    mu = np.nanmean(R)
    b_user = np.nanmean(R - mu, axis=1)
    residual = R - mu - b_user[:, None]

    n_item = known.sum(axis=0)
    b_item = np.zeros(R.shape[1])
    has_rating = n_item > 0
    b_item[has_rating] = (
        np.nansum(residual, axis=0)[has_rating] / n_item[has_rating])
    return mu, b_user, b_item

mu, b_user, b_item = bias_baseline(R)
pred = np.clip(mu + b_user[u] + b_item[i], 1, 5)
scores["user + movie effects"] = rmse(pred, truth)

print(pd.Series(scores, name="test RMSE").round(4).sort_values())

user + movie effects    0.9545
movie average           1.0284
user average            1.0426
overall average         1.1349
always 3                1.2516
Name: test RMSE, dtype: float64


---
## Practice Exercises

### Exercise 1: By hand, then by computer
Given the $3 \times 3$ ratings matrix with missing entries `?`:
$$R_{\text{tiny}} = \begin{bmatrix} 5 & 3 & ? \\ 4 & ? & 1 \\ ? & 2 & 2 \end{bmatrix}$$

**By hand calculation:**
1. **Overall mean ($\mu$):** There are 6 known ratings ($5, 3, 4, 1, 2, 2$):
   $$\mu = \frac{5 + 3 + 4 + 1 + 2 + 2}{6} = \frac{17}{6} \approx 2.8333$$
2. **User effects ($b_{\text{user}}$):** Each row's average minus $\mu$:
   - User 1 (row 1): $\frac{5 + 3}{2} - \frac{17}{6} = 4 - \frac{17}{6} = \frac{7}{6} \approx 1.1667$
   - User 2 (row 2): $\frac{4 + 1}{2} - \frac{17}{6} = 2.5 - \frac{17}{6} = -\frac{2}{6} = -\frac{1}{3} \approx -0.3333$
   - User 3 (row 3): $\frac{2 + 2}{2} - \frac{17}{6} = 2 - \frac{17}{6} = -\frac{5}{6} \approx -0.8333$
3. **Residuals ($R - \mu - b_{\text{user}}$):** Subtracting each user's row mean ($4, 2.5, 2$) from the known entries in that row gives:
   $$\text{residual} = \begin{bmatrix} 5 - 4 & 3 - 4 & ? \\ 4 - 2.5 & ? & 1 - 2.5 \\ ? & 2 - 2 & 2 - 2 \end{bmatrix} = \begin{bmatrix} 1 & -1 & ? \\ 1.5 & ? & -1.5 \\ ? & 0 & 0 \end{bmatrix}$$
4. **Movie effects ($b_{\text{item}}$):** Average the known residuals in each column:
   - Movie 1 (col 1): $\frac{1 + 1.5}{2} = 1.25$
   - Movie 2 (col 2): $\frac{-1 + 0}{2} = -0.50$
   - Movie 3 (col 3): $\frac{-1.5 + 0}{2} = -0.75$

Below we verify our hand calculations with `bias_baseline`:

In [7]:
R_tiny = np.array([
    [5.0, 3.0, np.nan],
    [4.0, np.nan, 1.0],
    [np.nan, 2.0, 2.0]
])

mu_tiny, b_user_tiny, b_item_tiny = bias_baseline(R_tiny)
print("mu:", round(mu_tiny, 4))
print("b_user:", np.round(b_user_tiny, 4))
print("b_item:", np.round(b_item_tiny, 4))

mu: 2.8333
b_user: [ 1.1667 -0.3333 -0.8333]
b_item: [ 1.25 -0.5  -0.75]


### Exercise 2: See what cheating looks like
Here we build a matrix from **all** 100,000 ratings (including the test set), compute user averages from it, and evaluate on the test set:

In [8]:
R_all = to_matrix(ratings)
user_mean_cheat = np.nanmean(R_all, axis=1)
cheat_rmse = rmse(user_mean_cheat[u], truth)

print(f"Honest user-average RMSE:   {scores['user average']:.4f}")
print(f"Cheating user-average RMSE: {cheat_rmse:.4f}")

Honest user-average RMSE:   1.0426
Cheating user-average RMSE: 1.0322


* **Explanation:** Yes, the cheating user-average RMSE (`1.0322`) is noticeably lower than the honest user-average RMSE (`1.0426`). This score is misleading because of **data leakage**: the hidden test ratings were included when calculating each user's average rating, pulling each user's mean toward the exact test ratings we are trying to predict. In a real recommendation system, future ratings are unknown at training time, so leaked scores give an artificially optimistic estimate of performance.

### Exercise 3: Is it luck?
We repeat the train/test split with `seed=1` and recompute all five baseline RMSE scores:

In [9]:
train_s1, test_s1 = split_ratings(ratings, test_frac=0.10, seed=1)
R_s1 = to_matrix(train_s1)
known_s1 = ~np.isnan(R_s1)

truth_s1 = test_s1["rating"].to_numpy()
u_s1 = test_s1["user"].to_numpy() - 1
i_s1 = test_s1["item"].to_numpy() - 1

mu_s1 = np.nanmean(R_s1)
user_mean_s1 = np.nanmean(R_s1, axis=1)
n_item_s1 = known_s1.sum(axis=0)
item_mean_s1 = np.full(N_ITEMS, mu_s1)
has_rating_s1 = n_item_s1 > 0
item_mean_s1[has_rating_s1] = (
    np.nansum(R_s1, axis=0)[has_rating_s1] / n_item_s1[has_rating_s1]
)

mu_b1, b_user_1, b_item_1 = bias_baseline(R_s1)
pred_b1 = np.clip(mu_b1 + b_user_1[u_s1] + b_item_1[i_s1], 1, 5)

scores_s1 = {
    "always 3": rmse(np.full(len(test_s1), 3.0), truth_s1),
    "overall average": rmse(np.full(len(test_s1), mu_s1), truth_s1),
    "user average": rmse(user_mean_s1[u_s1], truth_s1),
    "movie average": rmse(item_mean_s1[i_s1], truth_s1),
    "user + movie effects": rmse(pred_b1, truth_s1)
}

comparison = pd.DataFrame({"seed=0": scores, "seed=1": scores_s1}).round(4)
print(comparison.sort_values(by="seed=1"))

                      seed=0  seed=1
user + movie effects  0.9545  0.9577
movie average         1.0284  1.0159
user average          1.0426  1.0345
overall average       1.1349  1.1127
always 3              1.2516  1.2336


* **Explanation:** When we change the random split to `seed=1`, the individual RMSE numbers shift slightly (by about `0.002` to `0.008` stars) because a different random subset of ratings is held out for testing. However, the **ranking of the five models does not change at all** (`user + movie effects` is still best, followed by `movie average`, `user average`, `overall average`, and `always 3`). This shows that random sampling alone causes slight variations in RMSE, so we must always compare competing recommenders on the **exact same train/test split** rather than comparing scores computed on different splits.